# 🧠 Yüz Analizi — Derin Öğrenme Uygulaması

**Kuantum Bootcamp · Veysel Murat Görken**
*Data Scientist / AI Engineer*

---

Bu defter, sunumda anlattığımız kavramların **çalışan koddaki karşılığı**.
Sunumda slaytta gördüğünüz her fikir, burada birkaç satır Python olarak karşınıza çıkacak.

| Sunumda gördüğünüz | Defterde göreceğiniz |
|---|---|
| **CNN** — filtreler görüntünün üzerinde gezer | Yüzü bulan model bir CNN (MTCNN) |
| **Softmax** — olasılık dağılımı | Duygu tahmini 7 sınıfa dağılmış olasılık |
| **Embedding** — anlam bir vektöre dönüşür | Yüz de 512 boyutlu bir vektöre dönüşüyor |
| **Ölçekleme / normalizasyon** | Görüntü sabit boyuta getirilip normalize ediliyor |
| **Üretken vs ayırt edici modeller** | Üretilmiş bir yüzü, ayırt edici bir modele veriyoruz |

---

> ### ⚠️ Bu defterde **hiçbir şey eğitmiyoruz.**
> Tek bir `model.fit()` yok. Sadece **başkalarının eğitip yayınladığı** hazır modelleri
> indirip kullanıyoruz. Derin öğrenmenin günlük hayattaki kullanımının büyük kısmı
> tam olarak budur: sıfırdan eğitmek değil, hazır modeli doğru yerde kullanmak.

**Çalıştırma:** Hücreleri yukarıdan aşağıya sırayla çalıştırın (`Shift + Enter`).
GPU gerekmez — `Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU` yeterli.

---
## 0 · Kurulum

Colab'da **PyTorch, NumPy, Pillow, Matplotlib zaten kurulu**. Biz sadece iki şey ekliyoruz:

| Paket | Ne için |
|---|---|
| `facenet-pytorch==2.6.0` | Yüz bulma (MTCNN) + yüz vektörü (InceptionResnetV1) |
| `onnxruntime==1.24.1` | Duygu / yaş / cinsiyet modellerini çalıştırmak |

> **Konuşma notu:** "Buradaki en önemli detay `--no-deps`. Bu bayrak olmadan pip,
> Colab'daki PyTorch ve NumPy sürümlerini kendi istediğiyle değiştirmeye kalkar ve
> ortam bozulur. Kurulumda en sık yaşanan sorun budur."

**Neden OpenCV kurmuyoruz?** OpenCV 5 ile `cv2.CascadeClassifier` çekirdek modülden
kaldırıldı; eski API'ye dayanan yüz analizi kütüphaneleri bu yüzden kırılıyor.
Seçtiğimiz yığın OpenCV'ye **hiç ihtiyaç duymuyor** — bu kurulumu hem hızlı hem dayanıklı yapıyor.

In [ ]:
import os, sys, subprocess, warnings

# Log ve uyarı gürültüsünü baştan kısalım
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

ISARET_DOSYASI = "/tmp/.yuz_analizi_kurulum_yapildi"

def _pip(*argv):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *argv], check=False)

# facenet-pytorch: --no-deps => Colab'ın torch/numpy/pillow sürümlerine DOKUNMAZ
_pip("--no-deps", "facenet-pytorch==2.6.0")
# onnxruntime: hafif, OpenCV çekmez
_pip("onnxruntime==1.24.1")

# Kurulum sonrası import denemesi
try:
    import onnxruntime, facenet_pytorch  # noqa: F401
    print("✅ Kurulum tamam — runtime'ı yeniden başlatmaya GEREK YOK.")
except Exception as hata:
    if not os.path.exists(ISARET_DOSYASI):
        open(ISARET_DOSYASI, "w").write("1")     # döngüye girmemek için işaret bırak
        print("♻️  Runtime yeniden başlatılıyor... Bu hücre bittiğinde SONRAKİ hücreden devam edin.")
        import IPython
        IPython.Application.instance().kernel.do_shutdown(True)
    else:
        print("❌ Kurulum sonrası import hâlâ başarısız:", hata)

### 0.1 · Doğrulama

Kurulumun gerçekten çalıştığını **varsaymıyoruz, kontrol ediyoruz**: sürümleri yazdırıp
kritik fonksiyonların yerinde olup olmadığına bakıyoruz.

In [ ]:
import warnings, platform
warnings.filterwarnings("ignore")

import numpy as np, torch, PIL, matplotlib, onnxruntime as ort
import facenet_pytorch
from facenet_pytorch import MTCNN, InceptionResnetV1

print("Python          :", platform.python_version())
print("torch           :", torch.__version__)
print("numpy           :", np.__version__)
print("Pillow          :", PIL.__version__)
print("matplotlib      :", matplotlib.__version__)
print("onnxruntime     :", ort.__version__)
print("facenet-pytorch :", getattr(facenet_pytorch, "__version__", "2.6.0"))
print("-" * 46)

# Kritik fonksiyonlar gerçekten var mı?
kontroller = {
    "MTCNN.detect (yüz bulma)"          : hasattr(MTCNN, "detect"),
    "InceptionResnetV1 (yüz vektörü)"   : callable(InceptionResnetV1),
    "onnxruntime.InferenceSession"      : hasattr(ort, "InferenceSession"),
    "CPUExecutionProvider hazır"        : "CPUExecutionProvider" in ort.get_available_providers(),
}
for ad, sonuc in kontroller.items():
    print(("✅ " if sonuc else "❌ ") + ad)

assert all(kontroller.values()), "Bir bileşen eksik — 0. hücreyi tekrar çalıştırın."
print("-" * 46)
print("🎯 Ortam hazır. Hiçbir model EĞİTİLMEYECEK, sadece hazır modeller kullanılacak.")

### 0.2 · Ortak ayarlar

Sunumla aynı renk paletini kullanacağız ki grafikler slaytlarla aynı dili konuşsun.

In [ ]:
import matplotlib.pyplot as plt

ALTIN    = "#C08A2E"
LACIVERT = "#15263F"
TURKUAZ  = "#148F77"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.edgecolor":   LACIVERT,
    "axes.labelcolor":  LACIVERT,
    "text.color":       LACIVERT,
    "xtick.color":      LACIVERT,
    "ytick.color":      LACIVERT,
    "font.size":        12,
    "axes.titlesize":   14,
    "axes.titleweight": "bold",
})

CIHAZ = "cpu"
print("Palet ve grafik ayarları hazır · cihaz:", CIHAZ)

---
## 1 · Bir yüz bulalım

Analiz edeceğimiz fotoğrafı **üretken bir model** üretecek.
[thispersondoesnotexist.com](https://thispersondoesnotexist.com) her istekte,
dünyada var olmayan bir insanın yüzünü sıfırdan sentezleyen bir GAN çalıştırır.

> **Konuşma notu:** "Bu fotoğraftaki insan yok. Bir üretken model onu az önce uydurdu.
> Şimdi bu uydurma yüzü, ayırt edici bir modele vereceğiz. Sunumun başındaki
> *üretken vs ayırt edici* ayrımının canlı hâli bu: biri üretiyor, diğeri karar veriyor."

Bağlantı çalışmazsa yedek bir fotoğrafa düşeceğiz — dersin akışı bozulmasın.

In [ ]:
import io, urllib.request
from PIL import Image

BASLIK = {"User-Agent": "Mozilla/5.0 (Colab; Kuantum Bootcamp)"}

def internetten_resim_indir(url, zaman_asimi=30):
    """URL'den PIL görüntüsü indirir (User-Agent başlığı ile)."""
    istek = urllib.request.Request(url, headers=BASLIK)
    with urllib.request.urlopen(istek, timeout=zaman_asimi) as cevap:
        return Image.open(io.BytesIO(cevap.read())).convert("RGB")

YEDEK_URL = ("https://raw.githubusercontent.com/serengil/deepface/"
             "master/tests/unit/dataset/img4.jpg")

try:
    foto = internetten_resim_indir("https://thispersondoesnotexist.com/random-person.jpeg")
    kaynak = "Üretken model (GAN) tarafından sentezlendi — bu kişi gerçekte YOK"
except Exception as hata:
    print("⚠️  thispersondoesnotexist.com'a ulaşılamadı:", type(hata).__name__)
    print("    Yedek fotoğrafa geçiliyor. (İsterseniz 5. bölümde kendi fotoğrafınızı yükleyebilirsiniz.)")
    foto = internetten_resim_indir(YEDEK_URL)
    kaynak = "Yedek demo fotoğrafı"

plt.figure(figsize=(5, 5))
plt.imshow(foto); plt.axis("off")
plt.title("Analiz edeceğimiz fotoğraf", color=LACIVERT)
plt.show()

print("Kaynak :", kaynak)
print("Boyut  :", foto.size, "piksel (genişlik × yükseklik)")

---
## 2 · Adım 1 — Yüzü bul (*face detection*)

İlk iş: fotoğrafın **neresinde** yüz var?

Bunu **MTCNN** yapıyor — üç aşamalı, ardışık bir **CNN**. Sunumdaki "filtreler görüntünün
üzerinde gezer" animasyonunun tam karşılığı: küçük pencereler tüm görüntüyü tarar,
her aşama bir öncekinin elediği adayları rafine eder.

Model ağırlıkları paketin içinde geliyor — **indirme bile gerekmiyor.**

> **Konuşma notu:** "Dikkat edin: model fotoğraf görmüyor. Aşağıda yazdıracağımız gibi
> gördüğü şey `(3, 160, 160)` boyutunda bir sayı dizisi. Bizim 'yüz' dediğimiz şey
> onun için bir tensör."

In [ ]:
from facenet_pytorch import MTCNN
import matplotlib.patches as patches

# keep_all=True  -> fotoğraftaki TÜM yüzleri bulsun
# image_size=160 -> kırpılan yüz 160x160'a ölçeklensin (modelin beklediği boyut)
# margin=20      -> kutunun biraz dışını da al (saç/çene payı)
yuz_bulucu = MTCNN(image_size=160, margin=20, keep_all=True,
                   post_process=True, device=CIHAZ)

def yuzleri_bul(goruntu):
    """Görüntüdeki yüz kutularını ve güven skorlarını döndürür."""
    kutular, skorlar = yuz_bulucu.detect(goruntu)
    if kutular is None:
        return [], []
    return np.asarray(kutular, dtype=float), np.asarray(skorlar, dtype=float)

kutular, skorlar = yuzleri_bul(foto)
print(f"Bulunan yüz sayısı: {len(kutular)}")

# En güvenilir yüzü seçelim
en_iyi = int(np.argmax(skorlar))
kutu   = kutular[en_iyi]
print(f"Güven skoru       : {skorlar[en_iyi]:.4f}")
print(f"Kutu koordinatları: x1={kutu[0]:.0f}, y1={kutu[1]:.0f}, x2={kutu[2]:.0f}, y2={kutu[3]:.0f}")

# Modelin gerçekte gördüğü tensör
yuz_tensor = yuz_bulucu.extract(foto, kutular[en_iyi:en_iyi + 1], None)[0]

# --- Görselleştirme -------------------------------------------------------
fig, eksen = plt.subplots(1, 2, figsize=(11, 5.2))

eksen[0].imshow(foto); eksen[0].axis("off")
eksen[0].set_title("Orijinal + tespit kutusu")
for k, s in zip(kutular, skorlar):
    eksen[0].add_patch(patches.Rectangle((k[0], k[1]), k[2] - k[0], k[3] - k[1],
                                         linewidth=3, edgecolor=ALTIN, facecolor="none"))
    eksen[0].text(k[0], k[1] - 8, f"{s:.2f}", color="white", fontsize=11, fontweight="bold",
                  bbox=dict(facecolor=ALTIN, edgecolor="none", pad=2))

# Tensörü tekrar görüntüye çevirip gösterelim (sadece göz için)
gorsel = yuz_tensor.permute(1, 2, 0).numpy()
gorsel = (gorsel - gorsel.min()) / (gorsel.max() - gorsel.min())
eksen[1].imshow(gorsel); eksen[1].axis("off")
eksen[1].set_title("Kırpılmış + ölçeklenmiş yüz (160×160)")

plt.tight_layout(); plt.show()

# --- Sunumdaki "model sayı görür" mesajı ---------------------------------
print("\n" + "=" * 56)
print("MODELİN GÖRDÜĞÜ ŞEY")
print("=" * 56)
print("Dizi boyutu (kanal, yükseklik, genişlik):", tuple(yuz_tensor.shape))
print(f"Değer aralığı : [{yuz_tensor.min():.3f} , {yuz_tensor.max():.3f}]")
print(f"Ortalama      : {yuz_tensor.mean():.3f}")
print(f"Toplam sayı   : {yuz_tensor.numel():,} adet float")
print("\n👉 Model bir 'fotoğraf' görmüyor; sabit boyuta getirilmiş, normalize edilmiş")
print("   bir sayı dizisi görüyor. Sunumdaki ölçekleme/normalizasyon slaydı tam olarak buydu.")

---
## 3 · Adım 2 — Duygu analizi (*softmax*)

Şimdi kırptığımız yüzü bir **sınıflandırıcıya** veriyoruz. Model 7 duygu için birer skor
üretiyor, sonra **softmax** bu skorları **toplamı 1 olan bir olasılık dağılımına** çeviriyor.

Kullandığımız model: **HSEmotion `enet_b2_7`** (EfficientNet-B2, AffectNet üzerinde eğitilmiş).

> **Konuşma notu:** "Model 'mutlu' demiyor. 7 sayı veriyor ve bu 7 sayının toplamı tam olarak 1.
> 'Mutlu' dediğimiz şey, aslında bizim en yüksek olasılığı seçmemiz. Softmax slaydında
> gördüğünüz o dağılım, birazdan ekranda çubuk grafik olarak karşınıza çıkacak."

In [ ]:
import onnxruntime as ort

DUYGU_MODEL_URL = ("https://raw.githubusercontent.com/av-savchenko/face-emotion-recognition/"
                   "520a051c64cd191521e5934655314e769a319684/"
                   "models/affectnet_emotions/onnx/enet_b2_7.onnx")
DUYGU_MODEL_YOL = "/tmp/enet_b2_7.onnx"

# 7 sınıf — modelin çıktı sırası (İngilizce) ve Türkçe karşılıkları
DUYGU_SIRASI = ["Anger", "Disgust", "Fear", "Happiness", "Neutral", "Sadness", "Surprise"]
DUYGU_TR = {"Anger": "Kızgın", "Disgust": "İğrenme", "Fear": "Korku", "Happiness": "Mutlu",
            "Neutral": "Nötr", "Sadness": "Üzgün", "Surprise": "Şaşkın"}

if not os.path.exists(DUYGU_MODEL_YOL):
    print("Duygu modeli indiriliyor (~30 MB)...")
    urllib.request.urlretrieve(DUYGU_MODEL_URL, DUYGU_MODEL_YOL)

duygu_oturumu = ort.InferenceSession(DUYGU_MODEL_YOL, providers=["CPUExecutionProvider"])
print("✅ Duygu modeli hazır · girdi:", duygu_oturumu.get_inputs()[0].shape,
      "· çıktı:", duygu_oturumu.get_outputs()[0].shape)

IMAGENET_ORT = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def _yuzu_kirp(goruntu, kutu, buyutme=1.0, hedef=260):
    """Kutuyu (isteğe bağlı büyüterek) kırpar ve hedef boyuta ölçekler."""
    x1, y1, x2, y2 = kutu
    if buyutme != 1.0:
        mx, my = (x2 - x1) * (buyutme - 1) / 2, (y2 - y1) * (buyutme - 1) / 2
        x1, y1, x2, y2 = x1 - mx, y1 - my, x2 + mx, y2 + my
    return goruntu.crop((int(x1), int(y1), int(x2), int(y2))).resize((hedef, hedef), Image.BILINEAR)


def duygu_analiz_et(goruntu, kutu):
    """Yüz için 7 duyguya dağılmış olasılık vektörü döndürür."""
    kirpik = _yuzu_kirp(goruntu, kutu, buyutme=1.0, hedef=260)
    dizi = (np.asarray(kirpik, dtype=np.float32) / 255.0 - IMAGENET_ORT) / IMAGENET_STD
    dizi = dizi.transpose(2, 0, 1)[None]                      # (1, 3, 260, 260)
    skorlar = duygu_oturumu.run(None, {"input": dizi})[0][0]  # ham skorlar (logit)
    ustel = np.exp(skorlar - skorlar.max())                   # <-- SOFTMAX
    return ustel / ustel.sum()


olasiliklar = duygu_analiz_et(foto, kutu)

# --- Yatay bar grafik -----------------------------------------------------
sira = np.argsort(olasiliklar)
etiketler = [DUYGU_TR[DUYGU_SIRASI[i]] for i in sira]
degerler  = olasiliklar[sira]
renkler   = [ALTIN if i == len(sira) - 1 else LACIVERT for i in range(len(sira))]

plt.figure(figsize=(9, 4.5))
cubuklar = plt.barh(etiketler, degerler, color=renkler)
for cubuk, deger in zip(cubuklar, degerler):
    plt.text(deger + 0.012, cubuk.get_y() + cubuk.get_height() / 2,
             f"%{deger * 100:.1f}", va="center", fontsize=11, color=LACIVERT)
plt.xlim(0, 1.13)
plt.xlabel("Olasılık")
plt.title("Duygu dağılımı — softmax çıktısı (7 sınıf)")
plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# --- Softmax'in tanımını ekranda kanıtlayalım ----------------------------
kazanan = DUYGU_TR[DUYGU_SIRASI[int(np.argmax(olasiliklar))]]
print("=" * 56)
print(f"En yüksek olasılıklı duygu : {kazanan}  (%{olasiliklar.max() * 100:.1f})")
print("-" * 56)
for i in np.argsort(olasiliklar)[::-1]:
    print(f"  {DUYGU_TR[DUYGU_SIRASI[i]]:<10s} : {olasiliklar[i]:.6f}")
print("-" * 56)
print(f"OLASILIKLARIN TOPLAMI      : {olasiliklar.sum():.6f}")
print("👉 Toplam tam olarak 1. Softmax'in yaptığı iş buydu.")
print("=" * 56)

---
## 4 · Adım 3 — Yaş ve cinsiyet

Tek bir küçük model (InsightFace `genderage`, 1.3 MB) aynı anda iki iş yapıyor —
ve bu ikisi **farklı türde problemler**:

- **Yaş → regresyon.** Model tek bir *sayı* tahmin ediyor (ör. 33.7). Sınıf yok, süreklilik var.
- **Cinsiyet → sınıflandırma.** Model iki sınıfa dağılmış *olasılık* üretiyor, softmax ile.

> **Konuşma notu:** "Aynı ağın iki başı var. Biri sayı tahmin ediyor, diğeri sınıf seçiyor.
> Çıktı katmanına ne koyduğunuz, problemin türünü belirliyor."

Model 288 MB'lik bir arşivin içinde; bize sadece 1.3 MB'lik parça lazım.
HTTP *Range* isteğiyle **yalnızca o parçayı** indiriyoruz — arşivin tamamını çekmiyoruz.

In [ ]:
import struct, zlib

INSIGHTFACE_ZIP = "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"
YAS_MODEL_YOL   = "/tmp/genderage.onnx"


def _parca_indir(url, bas=None, son=None):
    baslik = dict(BASLIK)
    if bas is not None:
        baslik["Range"] = f"bytes={bas}-{son}"
    return urllib.request.urlopen(urllib.request.Request(url, headers=baslik), timeout=90)


def zipten_uye_indir(url, uye_adi, hedef):
    """Uzaktaki bir ZIP'in İÇİNDEN tek bir dosyayı, tamamını indirmeden çeker."""
    toplam = int(_parca_indir(url).headers["Content-Length"])
    kuyruk = _parca_indir(url, max(0, toplam - 65557), toplam - 1).read()
    i = kuyruk.rfind(b"PK\x05\x06")                                  # ZIP dizin sonu
    dizin_boyut, dizin_ofset = struct.unpack("<II", kuyruk[i + 12:i + 20])
    dizin = _parca_indir(url, dizin_ofset, dizin_ofset + dizin_boyut - 1).read()
    p = 0
    while p < len(dizin):
        yontem = struct.unpack("<H", dizin[p + 10:p + 12])[0]
        sikistirilmis = struct.unpack("<I", dizin[p + 20:p + 24])[0]
        n, e, c = struct.unpack("<HHH", dizin[p + 28:p + 34])
        yerel = struct.unpack("<I", dizin[p + 42:p + 46])[0]
        ad = dizin[p + 46:p + 46 + n].decode()
        if ad == uye_adi:
            basliklar = _parca_indir(url, yerel, yerel + 29).read()
            n2, e2 = struct.unpack("<HH", basliklar[26:30])
            basla = yerel + 30 + n2 + e2
            veri = _parca_indir(url, basla, basla + sikistirilmis - 1).read()
            if yontem == 8:
                veri = zlib.decompress(veri, -15)
            open(hedef, "wb").write(veri)
            return len(veri)
        p += 46 + n + e + c
    raise FileNotFoundError(uye_adi)


if not os.path.exists(YAS_MODEL_YOL):
    print("Yaş/cinsiyet modeli indiriliyor (arşivden sadece 1.3 MB)...")
    try:
        zipten_uye_indir(INSIGHTFACE_ZIP, "genderage.onnx", YAS_MODEL_YOL)
    except Exception as hata:                       # Range desteklenmezse tam arşive düş
        print("   Range isteği çalışmadı, arşivin tamamı indiriliyor...", type(hata).__name__)
        import zipfile
        urllib.request.urlretrieve(INSIGHTFACE_ZIP, "/tmp/buffalo_l.zip")
        with zipfile.ZipFile("/tmp/buffalo_l.zip") as z:
            open(YAS_MODEL_YOL, "wb").write(z.read("genderage.onnx"))

yas_oturumu = ort.InferenceSession(YAS_MODEL_YOL, providers=["CPUExecutionProvider"])
print("✅ Yaş/cinsiyet modeli hazır · girdi:", yas_oturumu.get_inputs()[0].shape)

CINSIYET_TR = {0: "Kadın", 1: "Erkek"}


def yas_cinsiyet_tahmin_et(goruntu, kutu):
    """(yaş, cinsiyet_etiketi, cinsiyet_olasiliklari) döndürür."""
    kirpik = _yuzu_kirp(goruntu, kutu, buyutme=1.5, hedef=96)   # model 96x96 ve geniş kırpım bekler
    dizi = np.asarray(kirpik, dtype=np.float32).transpose(2, 0, 1)[None]
    cikti = yas_oturumu.run(None, {"data": dizi})[0][0]         # [kadın_skor, erkek_skor, yaş/100]
    ustel = np.exp(cikti[:2] - cikti[:2].max())
    cinsiyet_olasilik = ustel / ustel.sum()                     # <-- yine softmax
    yas = float(cikti[2] * 100)                                 # <-- regresyon çıktısı
    return yas, CINSIYET_TR[int(np.argmax(cinsiyet_olasilik))], cinsiyet_olasilik


yas, cinsiyet, cinsiyet_olasilik = yas_cinsiyet_tahmin_et(foto, kutu)

print("=" * 56)
print(f"Tahmini yaş      : {yas:.1f}   → REGRESYON (tek bir sayı)")
print(f"Tahmini cinsiyet : {cinsiyet}  → SINIFLANDIRMA (olasılık dağılımı)")
print(f"   Kadın : %{cinsiyet_olasilik[0] * 100:.1f}")
print(f"   Erkek : %{cinsiyet_olasilik[1] * 100:.1f}")
print("=" * 56)

# --- Özet görsel ----------------------------------------------------------
fig, eksen = plt.subplots(1, 2, figsize=(10, 4.2),
                          gridspec_kw={"width_ratios": [1, 1.35]})
eksen[0].imshow(_yuzu_kirp(foto, kutu, 1.2, 200)); eksen[0].axis("off")
eksen[0].set_title(f"{cinsiyet} · ~{yas:.0f} yaş")

eksen[1].barh(["Kadın", "Erkek"], cinsiyet_olasilik, color=[TURKUAZ, LACIVERT])
eksen[1].set_xlim(0, 1.15); eksen[1].set_xlabel("Olasılık")
eksen[1].set_title("Cinsiyet — sınıflandırma")
for i, d in enumerate(cinsiyet_olasilik):
    eksen[1].text(d + 0.02, i, f"%{d * 100:.1f}", va="center", color=LACIVERT)
eksen[1].spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

---
## 5 · Kendi fotoğrafını dene

Şimdi 2, 3 ve 4. adımların hepsini **tek bir fonksiyonda** birleştiriyoruz.
Aşağıdaki hücre çalışınca bir dosya seçme düğmesi çıkacak — kendi fotoğrafınızı yükleyin.

> **Konuşma notu:** "Buraya kadar yaptığımız her şey aslında üç satır: yüzü bul, kırp,
> modele ver. Üretim kodunun büyük kısmı da böyle görünür."

Yüz bulunamazsa hata fırlatmıyoruz — anlaşılır bir uyarı verip devam ediyoruz.

In [ ]:
def tam_analiz(goruntu, baslik="Yüz Analizi"):
    """Bir görüntü için yüz tespiti + duygu + yaş/cinsiyet çalıştırır ve tek figürde gösterir."""
    kutular_, skorlar_ = yuzleri_bul(goruntu)

    if len(kutular_) == 0:
        print("⚠️  Bu fotoğrafta yüz bulunamadı.")
        print("    İpucu: yüz net ve öne dönük olsun, çok küçük/çok karanlık olmasın.")
        plt.figure(figsize=(4.5, 4.5))
        plt.imshow(goruntu); plt.axis("off"); plt.title("Yüz bulunamadı"); plt.show()
        return None

    en_iyi_ = int(np.argmax(skorlar_))
    kutu_   = kutular_[en_iyi_]

    olasilik_ = duygu_analiz_et(goruntu, kutu_)
    yas_, cinsiyet_, cins_olasilik_ = yas_cinsiyet_tahmin_et(goruntu, kutu_)
    duygu_ = DUYGU_TR[DUYGU_SIRASI[int(np.argmax(olasilik_))]]

    # --- tek figür, üç panel ---
    fig, eksen_ = plt.subplots(1, 3, figsize=(15, 5.0),
                               gridspec_kw={"width_ratios": [1, 1, 1.4]})
    fig.suptitle(baslik, fontsize=15, fontweight="bold", color=LACIVERT, y=1.02)

    eksen_[0].imshow(goruntu); eksen_[0].axis("off"); eksen_[0].set_title("Tespit")
    for k_ in kutular_:
        eksen_[0].add_patch(patches.Rectangle((k_[0], k_[1]), k_[2] - k_[0], k_[3] - k_[1],
                                              linewidth=3, edgecolor=ALTIN, facecolor="none"))

    eksen_[1].imshow(_yuzu_kirp(goruntu, kutu_, 1.2, 220)); eksen_[1].axis("off")
    eksen_[1].set_title(f"{cinsiyet_} · ~{yas_:.0f} yaş\nDuygu: {duygu_}")

    sira_ = np.argsort(olasilik_)
    renk_ = [ALTIN if j == len(sira_) - 1 else LACIVERT for j in range(len(sira_))]
    eksen_[2].barh([DUYGU_TR[DUYGU_SIRASI[i]] for i in sira_], olasilik_[sira_], color=renk_)
    eksen_[2].set_xlim(0, 1.1); eksen_[2].set_xlabel("Olasılık")
    eksen_[2].set_title("Duygu dağılımı (softmax)")
    eksen_[2].spines[["top", "right"]].set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

    print(f"Bulunan yüz sayısı : {len(kutular_)}")
    print(f"Duygu              : {duygu_} (%{olasilik_.max() * 100:.1f})")
    print(f"Yaş (regresyon)    : {yas_:.1f}")
    print(f"Cinsiyet (sınıf.)  : {cinsiyet_} (%{cins_olasilik_.max() * 100:.1f})")
    return {"kutu": kutu_, "duygu": duygu_, "yas": yas_, "cinsiyet": cinsiyet_}


print("✅ tam_analiz() hazır.")

In [ ]:
def resim_yukle():
    """Colab'da dosya yükleme penceresi açar; Colab dışında None döner."""
    try:
        from google.colab import files
    except ImportError:
        print("ℹ️  Bu hücre yalnızca Google Colab'da dosya yükleme açar.")
        return None
    yuklenen = files.upload()
    if not yuklenen:
        print("ℹ️  Dosya seçilmedi.")
        return None
    ad = list(yuklenen.keys())[0]
    print("Yüklenen dosya:", ad)
    return Image.open(io.BytesIO(yuklenen[ad])).convert("RGB")


kendi_foto = resim_yukle()

if kendi_foto is None:
    print("→ Demo fotoğrafıyla devam ediliyor.\n")
    tam_analiz(foto, baslik="Yüz Analizi — demo fotoğrafı")
else:
    tam_analiz(kendi_foto, baslik="Yüz Analizi — kendi fotoğrafınız")

---
## 6 · Bonus — Yüz de bir vektör (*embedding*)

Sunumda "kelimenin anlamı bir vektöre dönüşür" demiştik. Aynı fikir yüzler için de geçerli.

**InceptionResnetV1** (VGGFace2 üzerinde eğitilmiş) her yüzü **512 boyutlu bir vektöre**
çeviriyor. Bu vektör "kimlik"i taşıyor: aynı kişinin farklı fotoğrafları uzayda
birbirine yakın, farklı kişiler uzak düşüyor.

Yakınlığı **kosinüs benzerliği** ile ölçüyoruz: 1'e yakın = aynı yön = aynı kişi.

> **Konuşma notu:** "Burada hiçbir yerde 'bu Ahmet' yazmıyor. Model sadece sayı üretiyor.
> 'Aynı kişi mi?' sorusunu biz, iki sayı dizisi arasındaki açıya bakarak cevaplıyoruz.
> Embedding fikri tam olarak bu: anlam, bir sayı dizisine dönüşüyor."

In [ ]:
from facenet_pytorch import InceptionResnetV1

print("Yüz vektörü modeli yükleniyor (~107 MB, ilk seferde indirilir)...")
vektor_modeli = InceptionResnetV1(pretrained="vggface2").eval().to(CIHAZ)
print("✅ Hazır.")


def yuz_vektoru(goruntu):
    """Görüntüdeki en güvenilir yüzü 512 boyutlu vektöre çevirir."""
    kutular_, skorlar_ = yuzleri_bul(goruntu)
    if len(kutular_) == 0:
        return None
    en_iyi_ = int(np.argmax(skorlar_))
    tensor_ = yuz_bulucu.extract(goruntu, kutular_[en_iyi_:en_iyi_ + 1], None)
    with torch.no_grad():
        return vektor_modeli(tensor_.to(CIHAZ))[0].cpu().numpy()


vektor = yuz_vektoru(foto)
print("\n" + "=" * 56)
print("Vektör boyutu     :", vektor.shape, "→ 512 sayı")
print("İlk 8 eleman      :", np.round(vektor[:8], 4))
print(f"Vektör uzunluğu   : {np.linalg.norm(vektor):.4f}")
print("=" * 56)
print("👉 Bir yüz fotoğrafı, 512 sayıya indirgendi. Sunumdaki embedding slaydı buydu.")

### 6.1 · "Aynı kişi mi?"

Üç fotoğraf indiriyoruz: **A1** ve **A2** aynı kişinin iki farklı karesi, **B** başka biri.
Model bu kişileri hiç görmedi, isimlerini bilmiyor — sadece vektörleri karşılaştırıyoruz.

In [ ]:
def kosinus_benzerligi(a, b):
    """İki vektör arasındaki kosinüs benzerliği (-1 ile 1 arası)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


VERI = "https://raw.githubusercontent.com/serengil/deepface/master/tests/unit/dataset/"
KARSILASTIRMA = {"A1 (kişi 1)": VERI + "img1.jpg",
                 "A2 (kişi 1)": VERI + "img2.jpg",
                 "B  (kişi 2)": VERI + "img8.jpg"}

resimler, vektorler = {}, {}
for ad, url in KARSILASTIRMA.items():
    resimler[ad] = internetten_resim_indir(url)
    vektorler[ad] = yuz_vektoru(resimler[ad])

ESIK = 0.50   # facenet/VGGFace2 için yaygın bir kosinüs eşiği

fig, eksen = plt.subplots(1, 3, figsize=(11, 4))
for eks, (ad, resim) in zip(eksen, resimler.items()):
    eks.imshow(resim); eks.axis("off"); eks.set_title(ad)
plt.tight_layout(); plt.show()

print("=" * 62)
print(f"{'Karşılaştırma':<26s}{'Kosinüs':>10s}   Karar (eşik = %.2f)" % ESIK)
print("-" * 62)
adlar = list(vektorler)
for i in range(len(adlar)):
    for j in range(i + 1, len(adlar)):
        benzerlik = kosinus_benzerligi(vektorler[adlar[i]], vektorler[adlar[j]])
        karar = "✅ AYNI KİŞİ" if benzerlik >= ESIK else "❌ farklı kişi"
        print(f"{adlar[i]} ↔ {adlar[j]:<12s}{benzerlik:>8.3f}   {karar}")
print("=" * 62)
print("👉 Aynı kişinin iki karesi yüksek benzerlik, farklı kişi düşük benzerlik veriyor.")
print("   Model kimseyi 'tanımıyor'; sadece iki sayı dizisinin ne kadar aynı yöne")
print("   baktığına bakıyoruz. Yüz tanıma sistemlerinin temeli budur.")

---
## 7 · Sınırlar ve sorumluluk

Bu defter bir **demo**. Gerçek bir üründe kullanmadan önce bilinmesi gerekenler:

- **Modeller eğitim verisindeki dengesizlikleri taşır.** Bu modellerin eğitim setleri
  belirli yaş, etnik köken ve ışık koşullarına ağırlıklıdır. Performans herkes için
  eşit değildir ve bu bir "ayar" sorunu değil, verinin kendisiyle ilgilidir.
- **Yaş tahmini kaba bir tahmindir.** ±5–10 yıl sapma olağandır. Kimlik doğrulama veya
  yaş sınırı denetimi için kullanılamaz.
- **Duygu tahmini yüz ifadesini okur, kişinin gerçek duygusunu değil.** Gülümseyen biri
  mutlu olmayabilir. "Kızgın" etiketi bir iç dünya ölçümü değil, bir piksel örüntüsü tahminidir.
- **Cinsiyet sınıflandırması ikili bir varsayıma dayanır** ve bu varsayım gerçek insanları
  tam olarak temsil etmez.
- **Yüz verisi biyometrik veridir.** Rıza olmadan işlenmesi Türkiye'de **KVKK**,
  Avrupa'da **GDPR** kapsamında **özel nitelikli kişisel veri** sayılır. Demo için kendi
  fotoğrafınızı kullanın; başkasının fotoğrafını izinsiz yüklemeyin.
- **İşe alım, güvenlik, kredi, sigorta, eğitim değerlendirmesi gibi kararlarda
  kullanılmamalıdır.** Bu alanlarda hata bedeli insana ödetilir.

> **Konuşma notu:** "Bir modeli çalıştırabiliyor olmak, onu kullanmanın doğru olduğu
> anlamına gelmiyor. Teknik yapabilirlik ile etik uygunluk ayrı sorular ve ikincisini
> sormak da mühendisin işi."

---
## 🎯 Özet — sunumdaki kavramların defterdeki karşılığı

| Sunumdaki kavram | Bu defterde nerede gördük |
|---|---|
| **CNN** — filtreler görüntü üzerinde gezer | 2. bölüm: MTCNN yüzü buldu, kutuyu çizdi |
| **Ölçekleme / normalizasyon** | 2. bölüm: fotoğraf `(3, 160, 160)` tensöre dönüştü |
| **Softmax** — olasılık dağılımı | 3. bölüm: 7 duygu, toplamı tam olarak **1.000000** |
| **Regresyon vs sınıflandırma** | 4. bölüm: yaş bir sayı, cinsiyet bir dağılım |
| **Embedding** — anlam bir vektöre dönüşür | 6. bölüm: yüz → **512 boyutlu vektör** |
| **Üretken vs ayırt edici** | 1. bölüm: GAN üretti, sınıflandırıcılar yorumladı |
| **Sorumluluk** | 7. bölüm: yapabilmek ≠ yapmalı olmak |

---

### Ve tekrar: bu defterde **hiçbir model eğitilmedi.**

Tek bir `fit()` çağrısı yok. Başkalarının aylarca, binlerce GPU saati harcayarak eğittiği
modelleri indirip birleştirdik. Modern yapay zekâ mühendisliğinin büyük kısmı budur:
**doğru hazır parçayı, doğru yere takmak.**

---

*Kuantum Bootcamp · Deep Learning Master Class*
**Veysel Murat Görken** — Data Scientist / AI Engineer